# Inside `lab.go()`: optimizing a kernel for the board with ModelBlaster

`lab.go("maxpool2d_s8")` in `mb_lab.ipynb` runs a complete optimization in one call. This notebook runs the
same steps one command at a time, so you can see what each tool does and change any of it.

ModelBlaster compiles a PyTorch model into C for a RISC-V target: it lowers the model to an int8 graph and then
generates one kernel per operator. The kernels can come from a reference implementation, from a library of
handwritten kernels, or from an LLM, which ModelBlaster asks to write a kernel and then to make it faster. Every
candidate is checked bit for bit against the reference and timed on spike, an instruction set simulator. The
notebook adds the step that matters most for this board, hardware in the loop feedback: the LLM's kernel is
measured on the FPGA, and the measurement goes into the next round's prompt.

1. Lower the PyTorch op to an int8 graph (`extract_graph`).
2. Generate the reference kernel and time it on spike (`generate_kernels --backend reference`).
3. Let the LLM write and optimize a kernel (`generate_kernels --backend llm --optimize`).
4. Build both kernels for the board and measure them on the FPGA.
5. Give the LLM the FPGA's numbers (hardware in the loop feedback) and optimize again.
6. Build the best kernel with the MBP switched off, to see how much of the speedup is the accelerator.

The code cells are shell commands (`%%bash`), the same ones you would type in a terminal. Each run of the
setup cell creates a new folder under `by-hand/` for everything the later steps write; you can browse it on
the left. The notebook takes about 10 minutes (7 in our test run), most of it in the two LLM rounds.

> `mb_by_hand_solved.ipynb` is a recorded run of this notebook on a real board, with all of its output.

## 0. Setup

The next cell loads the seat's toolchain environment into this notebook (in a terminal you would run
`. ~/.config/iiswc/dev.env && . ~/iiswc-tutorial/env.sh`), picks the op, creates the run folder, and sets the
few variables ModelBlaster needs:

| variable | why |
|---|---|
| `PYTHONPATH` | ModelBlaster is run as `python -m modelblaster.pipeline.<tool>` from its checkout, `$ZCS` |
| `CPATH` | the kernels include `pext.h`, which exposes the MBP instructions as C functions |
| `MODELBLASTER_EXTRA_CMAKE_ARGS` | ModelBlaster times candidates on spike; this builds them without an FPU, like the board |
| `SPIKE_VERIFY_TIMEOUT` | a candidate that hangs on spike is dropped after 90 s instead of 300 s |
| `LLM_PROVIDER` | `--backend llm` calls `$MODEL` (DeepSeek) on AWS Bedrock with the seat's key |

In [ ]:
import os, subprocess, time
from pathlib import Path

env = subprocess.run(["bash", "-c", ". ~/.config/iiswc/dev.env && . ~/iiswc-tutorial/env.sh >/dev/null && env -0"],
                     capture_output=True, check=True).stdout.decode()
os.environ.update(line.split("=", 1) for line in env.split("\0") if "=" in line)

OP = "maxpool2d_s8"                                   # the op to optimize; section 1 lists the others
REPO = Path(os.environ["IISWC_ROOT"])
RUN = Path.home() / "work/modelblaster-llm-lab/by-hand" / time.strftime(f"{OP}-%m%d-%H%M%S")
(RUN / "logs").mkdir(parents=True)
os.chdir(REPO)                                        # the commands below use paths relative to the repository

os.environ.update({
    "OP": OP,
    "RUN": str(RUN),
    "PYTHONPATH": os.environ["ZCS"],
    "CPATH": str(REPO / "fpga/pynq-z2/sw"),
    "MODELBLASTER_EXTRA_CMAKE_ARGS": "-DCONFIG_FPU=n;-DCONFIG_FLOAT_HARD=n",
    "SPIKE_VERIFY_TIMEOUT": "90",
    "LLM_PROVIDER": "bedrock",
})
print("run folder:", RUN)

`mb doctor` checks the rest of the setup: the toolchain, the LLM key (with one short call), and your board. The
board keeps an ssh tunnel open to this seat, and the seat runs images on it through a small agent on the board.

In [ ]:
%%bash
mb doctor

## 1. The op

ModelBlaster starts from PyTorch. A bench file defines a `Model` and `get_inputs()`; this one is a single 2×2
max pool over a 16×64×64 int8 tensor.

In [ ]:
%%bash
cat fpga/pynq-z2/modelblaster/mb_ops/$OP.py

ModelBlaster knows how to generate a kernel for an op from its entry in `KERNEL_SPECS`
(`modelblaster/pipeline/reference_kernels.py`). An entry holds the C signature, a plain reference implementation
that every generated kernel is checked against, and a list of algorithms. An algorithm is a short description,
with an example, that the LLM is given when it writes the kernel; each one names the targets it suits. The next
cell lists the ops that have an algorithm for this board's target, `pext` (the Rocket core with the MBP), and
prints the description the LLM will get for `maxpool2d_s8`.

To optimize another op, set `OP` in the setup cell to `linear_s8` or `gelu_s8` (they have bench files here) and
run the notebook again from the top.

In [ ]:
%%bash
python - <<'EOF'
import os
from modelblaster.pipeline.reference_kernels import KERNEL_SPECS

for op, spec in sorted(KERNEL_SPECS.items()):
    algorithms = [a.name for a in spec.algorithms if "pext" in a.target_affinity]
    if algorithms:
        print(f"{op:16} {', '.join(algorithms)}")

for a in KERNEL_SPECS[os.environ["OP"]].algorithms:
    if "pext" in a.target_affinity:
        print(f"\n{a.name}:\n{a.description}")
EOF

## 2. Lower the op to an int8 graph

`extract_graph` runs the model once, quantizes it to int8 with one calibration batch, and fuses what the target
can fuse. It writes the graph (`graph.json`: ops, shapes and scales), the weights, and `io.npz`: an input and
PyTorch's output for it, which every generated kernel has to reproduce exactly.

In [ ]:
%%bash
python -m modelblaster.pipeline.extract_graph --bench-file fpga/pynq-z2/modelblaster/mb_ops/$OP.py \
    --out-dir $RUN/ir --quant int8 --num-calibration 1 --fusion-target pext > $RUN/logs/extract_graph.log
ls $RUN/ir

In [ ]:
import json

graph = json.loads((RUN / "ir/graph.json").read_text())
for op in graph["ops"]:
    print(op["op"], op.get("shape"))

## 3. The reference kernel

`generate_skeleton` writes the C for the model around its kernels: buffers, weights and the test data.
`generate_kernels` then fills in one kernel per op. It looks in a library of handwritten kernels first
(`--global-curated-dir`); a kernel found there is used as is, and the log says `curated HIT`. Otherwise it uses the
op's reference implementation (`--backend reference`) or asks the LLM (`--backend llm`).

This repository's library already has a fast `maxpool2d_s8`, so the run works on a copy with that kernel removed:
the reference kernel is then ModelBlaster's plain C, and a faster one has to come from the LLM. The run also gets
a copy of ModelBlaster's spike harness, which lacks a configuration file for the `pext` target.

In [ ]:
%%bash
cp -r fpga/pynq-z2/modelblaster/kernels $RUN/curated
rm $RUN/curated/pext*/pext*_${OP}_*.c                 # remove the handwritten kernels for this op
cp -r $ZCS/modelblaster/harness $RUN/harness && chmod -R u+w $RUN/harness
touch $RUN/harness/backends/pext.conf

In [ ]:
%%bash
python -m modelblaster.pipeline.generate_skeleton --ir $RUN/ir/graph.json --weights $RUN/ir/weights.npz \
    --io $RUN/ir/io.npz --backend pext --out-dir $RUN/reference > $RUN/logs/skeleton-reference.log
cp $RUN/ir/graph.json $RUN/reference/                # the harness build reads the graph from here

python -m modelblaster.pipeline.generate_kernels --ir $RUN/ir/graph.json --io $RUN/ir/io.npz \
    --target pext --quant int8 --backend reference --global-curated-dir $RUN/curated \
    --out-dir $RUN/reference --cache-dir $RUN/cache-reference --build-dir $RUN/build/candidates \
    --repo-root $ZCS/modelblaster --harness-dir $RUN/harness > $RUN/logs/generate-reference.log 2>&1
cat $RUN/reference/kernel_picks.json

To time the reference kernel, build ModelBlaster's harness for spike with `west`, Zephyr's build tool, and run it.
`-DMB_PEXT_HW=1` compiles the MBP functions in `pext.h` to the real instructions. The image runs the model once,
compares the output with PyTorch's (`MODELBLASTER_VERIFY`) and prints one line per op with its cycle count. Spike
counts one cycle per instruction.

In [ ]:
%%bash
west build -p always -b spike_riscv64 $RUN/harness -d $RUN/build/spike-reference -- \
    -DMODEL_DIR=$RUN/reference -DMODELBLASTER_BACKEND=pext -DMODELBLASTER_KERNEL_CFLAGS=-DMB_PEXT_HW=1 \
    -DCONFIG_FPU=n -DCONFIG_FLOAT_HARD=n > $RUN/logs/build-spike-reference.log 2>&1
spike $RUN/build/spike-reference/zephyr/zephyr.elf | tee $RUN/spike-reference.txt | grep -E "VERIFY|,$OP,"

## 4. The LLM writes a kernel, and optimizes it

With `--backend llm`, `generate_kernels` gives the LLM the op's reference implementation and an algorithm
description, and asks for a kernel. Each answer is compiled here and checked against the reference, bit for bit;
if it fails, the error goes back to the LLM. `--optimize` then runs a small beam search: it times the first
correct kernel on spike, asks the LLM for `--expansions` faster variants of each of the `--beam` best kernels,
checks and times each, and keeps the fastest. Every correct kernel is stored in `--cache-dir`.

ModelBlaster's prompt does not describe the MBP, so the call goes through `scripts/lib/mb_llm_tap.py`, a small
wrapper that passes everything after `--` to `generate_kernels` unchanged. Each `--system-append` file is added to
ModelBlaster's system prompt: here the MBP instruction guide (`pext_isa_guide.md`, what the four MBP
instructions do) and `idea_line.md` (a rule asking the LLM to state each candidate's idea in one line). The
wrapper also records every prompt and answer in `llm-calls.jsonl` and stops after `--max-calls`. ModelBlaster's
own output goes to the log, and the cell shows one line per LLM call. A round takes one to three minutes.

In [ ]:
%%bash
python -m modelblaster.pipeline.generate_skeleton --ir $RUN/ir/graph.json --weights $RUN/ir/weights.npz \
    --io $RUN/ir/io.npz --backend pext --out-dir $RUN/round1 > $RUN/logs/skeleton-round1.log
cp $RUN/ir/graph.json $RUN/round1/

python scripts/lib/mb_llm_tap.py --transcript $RUN/llm-calls.jsonl --max-calls 6 --round 1 \
    --system-append fpga/pynq-z2/modelblaster/mb_ops/pext_isa_guide.md \
    --system-append fpga/pynq-z2/modelblaster/mb_ops/idea_line.md \
    --log $RUN/logs/generate-round1.log -- \
  --ir $RUN/ir/graph.json --io $RUN/ir/io.npz --target pext --quant int8 \
  --backend llm --optimize --beam 2 --expansions 2 --global-curated-dir $RUN/curated \
  --out-dir $RUN/round1 --cache-dir $RUN/cache --build-dir $RUN/build/candidates \
  --repo-root $ZCS/modelblaster --harness-dir $RUN/harness

The kernel ModelBlaster kept is the newest file in the cache. The lines that call `mb_pext_max8` are the MBP
instruction.

In [ ]:
kernel = max((RUN / "cache").glob("*.c"), key=os.path.getmtime)
print(kernel.name, "\n")
print(kernel.read_text())

Every call to the LLM, from `llm-calls.jsonl`, with the start of the first prompt (what ModelBlaster asks for) and of its answer:

In [ ]:
calls = [json.loads(line) for line in (RUN / "llm-calls.jsonl").read_text().splitlines()]
for c in calls:
    added = [name for name, mark in [("MBP instruction guide", "packed-SIMD integer extension (MBP)"),
                                      ("hardware in the loop feedback", "### Hardware in the loop feedback")]
             if mark in (c["system"] or "")]
    print(f"#{c['n']}  round {c['round']}  {c['phase']:<24} {c['latency_s']:4.0f} s  "
          f"{c['input_tokens']:,} tokens in, {c['output_tokens']:,} out   added: {', '.join(added)}")

print("\n--- prompt of call #1 (first 25 lines) ---")
print("\n".join(calls[0]["user"].splitlines()[:25]))
print("\n--- answer of call #1 (first 15 lines) ---")
print("\n".join((calls[0]["response"] or "").splitlines()[:15]))

## 5. Measure on the FPGA

The board runs a fixed bitstream: a Rocket core at 40 MHz with the MBP on hart 0 (MAGIC `0x5A5A0038`). Images for
it use Zephyr's board `chipyard_pynqz1_all_f40` and this repository's sample `samples/modelblaster_pext`, which
runs the model on hart 0 `MB_ITERS` times after one warmup, checks the output against PyTorch's, and reports the
cycles of each op. The next cell builds the reference and the round-1 kernel.

In [ ]:
%%bash
for model in reference round1; do
  west build -p always -b chipyard_pynqz1_all_f40 samples/modelblaster_pext -d $RUN/build/board-$model -- \
      -DBOARD_ROOT=$PWD -DMODEL_DIR=$RUN/$model -DMODELBLASTER_KERNEL_CFLAGS=-DMB_PEXT_HW=1 -DMB_ITERS=3 \
      > $RUN/logs/build-board-$model.log 2>&1
done
cd $RUN/build && ls -l board-*/zephyr/zephyr.bin

`mb run-image` runs one image on your board and prints its console. It uses the board agent's three commands
(`put` the image, `run` it, `get` the console) and waits if another run is using your board:

    ssh -p 19022 -i ~/.ssh/iiswc-board-agent xilinx@localhost "put zephyr.bin <bytes> <md5>" < zephyr.bin
    ssh -p 19022 -i ~/.ssh/iiswc-board-agent xilinx@localhost "run zephyr"
    ssh -p 19022 -i ~/.ssh/iiswc-board-agent xilinx@localhost "get console.out"

Each run takes about 30 seconds.

In [ ]:
%%bash
mkdir -p $RUN/fpga
mb run-image $RUN/build/board-reference/zephyr/zephyr.bin > $RUN/fpga/reference.txt
mb run-image $RUN/build/board-round1/zephyr/zephyr.bin > $RUN/fpga/round1.txt
grep -h -E "^(MB_PEXT_OP|RESULT)" $RUN/fpga/reference.txt $RUN/fpga/round1.txt

In [ ]:
import re

def fpga(model):
    """The op's cycles on the FPGA, and cycles per output, from a board console."""
    text = (RUN / f"fpga/{model}.txt").read_text(errors="replace")
    cycles = int(re.search(rf"^MB_PEXT_OP .* op={OP} .*cycles=(\d+)", text, re.M).group(1))
    outputs = len(re.search(r"^MB_PEXT_OUT (.*)$", text, re.M).group(1).split())
    return cycles, cycles / outputs

reference, round1 = fpga("reference"), fpga("round1")
print(f"reference kernel   {reference[1]:6.1f} cycles per output")
print(f"round 1 kernel     {round1[1]:6.1f} cycles per output   {reference[0] / round1[0]:.1f}x faster on your FPGA")

## 6. Hardware in the loop feedback: tell the LLM what the FPGA measured, and optimize again

Spike charges one cycle per instruction and does not model memory, so a kernel that is fast on spike can be
slower than expected on the board. `lab.go` therefore runs each round's best kernel on the FPGA and adds the
measurement to the next round's system prompt. This is the only information the LLM gets that ModelBlaster and
the MBP instruction guide do not give it. Here is that step by hand: write the numbers to a file, and run round 2
with the file as one more `--system-append`. Round 2 uses the same `--cache-dir`, so it starts from
round 1's kernel. When round 1 is already close to what the board allows, round 2 may not find anything faster;
ModelBlaster then keeps round 1's kernel.

In [ ]:
feedback = RUN / "fpga-feedback.md"
feedback.write_text(f"""### Hardware in the loop feedback: cycles measured on the FPGA
Round 1's kernel was run on the real board (a Rocket core at 40 MHz with the MBP; cycles from rdcycle).
Spike, which scores your candidates, charges one cycle per instruction and has no memory timing, so a kernel
with fewer, wider memory accesses gains more on the board than spike shows. Optimize for these numbers:
- reference kernel: {reference[1]:.1f} cycles per output
- round 1's kernel: {round1[1]:.1f} cycles per output ({reference[0] / round1[0]:.1f}x faster than the reference)
""")
print(feedback.read_text())

In [ ]:
%%bash
python -m modelblaster.pipeline.generate_skeleton --ir $RUN/ir/graph.json --weights $RUN/ir/weights.npz \
    --io $RUN/ir/io.npz --backend pext --out-dir $RUN/round2 > $RUN/logs/skeleton-round2.log
cp $RUN/ir/graph.json $RUN/round2/

python scripts/lib/mb_llm_tap.py --transcript $RUN/llm-calls.jsonl --max-calls 6 --round 2 \
    --system-append fpga/pynq-z2/modelblaster/mb_ops/pext_isa_guide.md \
    --system-append fpga/pynq-z2/modelblaster/mb_ops/idea_line.md \
    --system-append $RUN/fpga-feedback.md \
    --log $RUN/logs/generate-round2.log -- \
  --ir $RUN/ir/graph.json --io $RUN/ir/io.npz --target pext --quant int8 \
  --backend llm --optimize --beam 2 --expansions 2 --global-curated-dir $RUN/curated \
  --out-dir $RUN/round2 --cache-dir $RUN/cache --build-dir $RUN/build/candidates \
  --repo-root $ZCS/modelblaster --harness-dir $RUN/harness

In [ ]:
%%bash
west build -p always -b chipyard_pynqz1_all_f40 samples/modelblaster_pext -d $RUN/build/board-round2 -- \
    -DBOARD_ROOT=$PWD -DMODEL_DIR=$RUN/round2 -DMODELBLASTER_KERNEL_CFLAGS=-DMB_PEXT_HW=1 -DMB_ITERS=3 \
    > $RUN/logs/build-board-round2.log 2>&1
mb run-image $RUN/build/board-round2/zephyr/zephyr.bin > $RUN/fpga/round2.txt
grep -E "^(MB_PEXT_OP|RESULT)" $RUN/fpga/round2.txt

In [ ]:
round2 = fpga("round2")
for name, (cycles, per_output) in [("reference", reference), ("round 1", round1), ("round 2", round2)]:
    print(f"{name:10} {per_output:6.1f} cycles per output   {reference[0] / cycles:5.1f}x")

best = "round2" if round2[0] < round1[0] else "round1"
os.environ["BEST"] = best
print("\nfastest on the FPGA:", best)

## 7. How much of the speedup is the accelerator?

`objdump` shows the MBP instructions in the best kernel: the assembler does not know them, so they appear as
`.insn` words. Then the same kernel is built with `-DMB_PEXT_HW=0`, which makes `pext.h` compile each MBP
instruction to its C equivalent, and run on the board again. The difference between the two is what the
accelerator contributes.

In [ ]:
%%bash
riscv64-zephyr-elf-objdump -d --disassemble=kernel_${OP}_kb_${OP} $RUN/build/board-$BEST/zephyr/zephyr.elf \
    | grep insn || echo "no MBP instructions in this kernel"

west build -p always -b chipyard_pynqz1_all_f40 samples/modelblaster_pext -d $RUN/build/board-mbpoff -- \
    -DBOARD_ROOT=$PWD -DMODEL_DIR=$RUN/$BEST -DMODELBLASTER_KERNEL_CFLAGS=-DMB_PEXT_HW=0 -DMB_ITERS=3 \
    > $RUN/logs/build-board-mbpoff.log 2>&1
mb run-image $RUN/build/board-mbpoff/zephyr/zephyr.bin > $RUN/fpga/mbpoff.txt
grep -E "^(MB_PEXT_OP|RESULT)" $RUN/fpga/mbpoff.txt

## 8. Results

The speedup on the FPGA splits into two factors that multiply: the rewritten loop (the reference against the
LLM's kernel with the MBP off) and the accelerator (the LLM's kernel with the MBP off against on). `lab.verdict()`
in `mb_lab.ipynb` reports the same numbers.

In [ ]:
spike_reference = int(re.search(rf"^\d+,\w+,{OP},[^,]*,(\d+)$", (RUN / "spike-reference.txt").read_text(), re.M).group(1))
spike = {"reference": spike_reference}
for r in ("round1", "round2"):
    spike[r] = json.loads((RUN / r / "optimize_summary.json").read_text())[OP]["best"]
board = {"reference": reference, "round1": round1, "round2": round2, "mbpoff": fpga("mbpoff")}
outputs = reference[0] / reference[1]

print(f"{'':22}{'spike':>10}{'FPGA':>10}   cycles per output")
for name in ("reference", "round1", "round2"):
    print(f"{name:22}{spike[name] / outputs:10.1f}{board[name][1]:10.1f}")
print(f"{best + ', MBP off':22}{'':>10}{board['mbpoff'][1]:10.1f}")

total = reference[0] / board[best][0]
loop = reference[0] / board["mbpoff"][0]
accelerator = board["mbpoff"][0] / board[best][0]
print(f"\n{total:.1f}x faster on your FPGA = {loop:.1f}x from the rewritten loop x {accelerator:.1f}x from the accelerator")

## 9. The same thing in one command

`mb go maxpool2d_s8` in a terminal, or `lab.go("maxpool2d_s8")` in `mb_lab.ipynb`, runs these steps. It also
checks exactness over the whole input range where the op allows it, falls back to a recorded kernel when the
LLM is unavailable and to spike alone when the board is, and writes a report. `mb commands <run>` lists every
command a run executed.

## 10. Exercises

1. Run `generate_kernels --backend llm` with the full library of handwritten kernels. The next cell does this;
   look for `curated HIT` in its output. Was the LLM called?
2. Run round 1 again without the MBP instruction guide (remove its `--system-append` line), with a new
   `--cache-dir` and `--out-dir`. Does the LLM find MBP.MAX8 from ModelBlaster's own algorithm description?
3. Compare spike's and the FPGA's cycles per output in section 8. Why is the gap larger for the LLM's kernels
   than for the reference?
4. Set `OP = "linear_s8"` or `OP = "gelu_s8"` in the setup cell and run the notebook again.

In [ ]:
%%bash
cp -r $RUN/reference $RUN/with-curated
python -m modelblaster.pipeline.generate_kernels --ir $RUN/ir/graph.json --io $RUN/ir/io.npz \
    --target pext --quant int8 --backend llm --global-curated-dir fpga/pynq-z2/modelblaster/kernels \
    --out-dir $RUN/with-curated --cache-dir $RUN/cache-with-curated --build-dir $RUN/build/candidates \
    --repo-root $ZCS/modelblaster --harness-dir $RUN/harness > $RUN/logs/generate-with-curated.log 2>&1
grep "curated HIT" $RUN/logs/generate-with-curated.log
cat $RUN/with-curated/kernel_picks.json